In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, recall_score
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np


In [13]:
def categorize_conflict(x, q1, q3):
    if x == 0:
        return 0
    elif x <= q1:
        return 1
    elif x <= q3:
        return 2
    else:
        return 3

In [14]:
parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

#Drop nas #
df = pd.read_parquet(parquet_path)
df = df.dropna()
new_df = df[df["conflict_count"] > 0]
q1, q2, q3 = new_df['conflict_count'].quantile(0.25), new_df['conflict_count'].quantile(0.50), new_df['conflict_count'].quantile(0.75)

df['target'] = df['conflict_count'].apply(lambda x: categorize_conflict(x, q1, q3))
features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
features = pd.get_dummies(features, columns=['year'], prefix='year')
X = features
y = df['target']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

In [17]:
param_grid = {
    'n_neighbors': [3,5,7,9,11],
    'weights': ['distance', 'uniform'], 
    'metric': ['euclidean']
}

In [18]:
knn = KNeighborsClassifier()

grid_knn = GridSearchCV(
    knn, 
    param_grid,
    cv = 5, 
    scoring='recall_weighted',
    verbose=1,
    n_jobs=-1
)


In [19]:
grid_knn.fit(X_train_balanced, y_train_balanced)
best_knn = grid_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [20]:
print("\n--- Tuned KNN Model ---")
print(f"Best Parameters: {grid_knn.best_params_}")
print(f"\nTest Accuracy: {best_knn.score(X_test, y_test):.4f}")
print(f"Best Weighted Recall (CV): {grid_knn.best_score_:.4f}")
print("\nTest Recall (weighted):", recall_score(y_test, y_pred_knn, average='weighted'))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))


--- Tuned KNN Model ---
Best Parameters: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}

Test Accuracy: 0.8235
Best Weighted Recall (CV): 0.9580

Test Recall (weighted): 0.8234602381302366

Confusion Matrix:
[[5418  446  296   76]
 [  92   57   57   23]
 [  43   46   70   41]
 [  16   18   47   57]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.87      0.92      6236
           1       0.10      0.25      0.14       229
           2       0.15      0.35      0.21       200
           3       0.29      0.41      0.34       138

    accuracy                           0.82      6803
   macro avg       0.38      0.47      0.40      6803
weighted avg       0.91      0.82      0.86      6803

